# Exploration du data lake et du data warehouse

Même connexion que le DAG Airflow : DuckDB lit le bucket S3 et interroge PostgreSQL.

In [1]:
import os
import duckdb
import graphiques  # commun/graphiques.py : les mêmes graphiques que le site web

con = duckdb.connect()
con.execute("LOAD httpfs; LOAD postgres")
con.execute(f"""
    CREATE SECRET (TYPE s3, KEY_ID '{os.environ['S3_ACCESS_KEY']}', SECRET '{os.environ['S3_SECRET_KEY']}',
                   ENDPOINT '{os.environ['S3_ENDPOINT']}', URL_STYLE 'path', USE_SSL false, REGION 'us-east-1')
""")
con.execute(f"ATTACH '{os.environ['DW_DSN']}' AS dw (TYPE postgres, READ_ONLY)")
BUCKET = f"s3://{os.environ['S3_BUCKET']}"

## 1. Le data lake

`raw/` : fichiers bruts déposés au démarrage ; `clean/` : fichiers nettoyés par le DAG.

In [2]:
con.sql(f"SELECT file_name, num_rows AS lignes FROM parquet_file_metadata('{BUCKET}/**/*.parquet')")

┌────────────────────────────────────────────────────┬─────────┐
│                     file_name                      │ lignes  │
│                      varchar                       │  int64  │
├────────────────────────────────────────────────────┼─────────┤
│ s3://datalake/clean/annonces.parquet               │ 9365126 │
│ s3://datalake/clean/bilans.parquet                 │ 6336013 │
│ s3://datalake/raw/bodacc/creations.parquet         │ 5160000 │
│ s3://datalake/raw/bodacc/radiations.parquet        │ 4272554 │
│ s3://datalake/raw/inpi/donnees_financieres.parquet │ 6368964 │
└────────────────────────────────────────────────────┴─────────┘

## 2. Prototyper une règle avant de la mettre dans le DAG : le SIREN

Dans le BODACC, le champ `registre` contient le SIREN deux fois, avec et sans espaces, dans un ordre qui varie. On teste l'expression sur des cas connus, puis sur les vraies données.

In [3]:
SIREN = "nullif(regexp_extract(replace(split_part(registre, ',', 1), ' ', ''), '^[0-9]{9}$'), '')"

con.sql(f"""
    SELECT registre, {SIREN} AS siren
    FROM (VALUES ('449 850 908,449850908'), ('449850908,449 850 908'), ('RCS Paris'), (NULL)) t(registre)
""")

┌───────────────────────┬───────────┐
│       registre        │   siren   │
│        varchar        │  varchar  │
├───────────────────────┼───────────┤
│ 449 850 908,449850908 │ 449850908 │
│ 449850908,449 850 908 │ 449850908 │
│ RCS Paris             │ NULL      │
│ NULL                  │ NULL      │
└───────────────────────┴───────────┘

In [4]:
con.sql(f"""
    SELECT count(*) AS annonces,
           count({SIREN}) AS siren_valide,
           round(100 * count({SIREN}) / count(*), 1) AS pct_valide
    FROM read_parquet('{BUCKET}/raw/bodacc/creations.parquet')
    WHERE typeavis = 'annonce'
""")

┌──────────┬──────────────┬────────────┐
│ annonces │ siren_valide │ pct_valide │
│  int64   │    int64     │   double   │
├──────────┼──────────────┼────────────┤
│  5150236 │      5071774 │       98.5 │
└──────────┴──────────────┴────────────┘

## 3. Data warehouse : sociétés créées et radiées par année

Les radiations d'office (nettoyage administratif du registre) sont séparées des autres radiations.

In [ ]:
graphiques.creations_radiations(con)

## 4. Data warehouse : impôt sur les bénéfices par taille d'entreprise (2023)

In [ ]:
graphiques.is_par_tranche(con)